In [ ]:
%load_ext autoreload
%autoreload 2
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.abspath(".."))

import src.spaces as spaces
spaces.SETUP = spaces.make_setup_matrix()

from src.training import (
    train_resnet,
    loss_fn_da_resnet,
    loss_fn_fa_resnet,
    DEFAULT_CONFIG_RESNET,
)
from src.teacher import make_wi_particles_resnet
from src.utils import plot_training_curves, plot_loss_curves, DEFAULT_MARKERS, DEFAULT_COLORS

# Hyperparameter tuning — ResNet, matrix setup

**Observed problem**: the loss *increases* with the number of SGD steps.

**Likely cause**: in the matrix ResNet, each layer computes
`h_l = h_{l-1} + (α_arch/L)·(1/M)·Σ sigmoid(z·h_{l-1})`.  
During training the particles `z` grow, and the (unnormalised) pre-activation `z·h`,
with `h` ~ σ_π·O(1) = O(4), leaves the linear regime of the sigmoid. The effective
gradient then grows with `z`, and the SGD step `α·∇L` can overshoot → divergence.

This notebook:
1. **Diagnostic**: visually confirms at which L the instability appears
2. **Grid search α × α_arch**: the two main stability knobs
3. **τ sweep**: checks the effect of the regularization
4. **Full validation**: vanilla / DA / FA with the best configuration

---
## 1. Diagnostic — curves with DEFAULT_CONFIG_RESNET

We plot the vanilla training loss for several values of L,
with the default config (`α=5`, `α_arch=1`), to locate where it diverges.

In [ ]:
M_DIAG = 5
L_DIAG_VALUES = [1, 10, 100, 500]

teacher_diag = make_wi_particles_resnet(1)
CONFIG_DIAG  = {**DEFAULT_CONFIG_RESNET, "T": 2.0, "gr": 20, "beta": 0.0}

fig, axes = plt.subplots(1, len(L_DIAG_VALUES), figsize=(14, 4), sharey=False)

for ax, L in zip(axes, L_DIAG_VALUES):
    key = jax.random.PRNGKey(42)
    h   = train_resnet(teacher_diag, M_DIAG, L, key, CONFIG_DIAG, "si")
    steps  = np.array(h["steps"][1:])
    losses = np.array(h["losses"])
    color  = "crimson" if losses[-1] > losses[0] else "steelblue"
    ax.plot(steps, losses, marker="o", markersize=3, color=color)
    ax.set_title(f"L={L}  {'⚠ diverge' if losses[-1] > losses[0] else '✓ ok'}", fontsize=11)
    ax.set_xlabel("SGD step")
    ax.set_ylabel("Loss")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    print(f"L={L:4d}  loss_init={losses[0]:.4f}  loss_final={losses[-1]:.4f}  "
          f"ratio={losses[-1]/losses[0]:.2f}")

fig.suptitle(f"DEFAULT_CONFIG_RESNET (α=5, α_arch=1) — M={M_DIAG}", fontsize=12)
fig.tight_layout()
plt.show()

---
## 2. Grid search α × α_arch

For `L=100` (the problematic case), we sweep:
- `alpha`      ∈ {0.01, 0.05, 0.1, 0.5, 1.0, 5.0}  (DEFAULT = 5.0)
- `alpha_arch` ∈ {0.1, 0.5, 1.0, 2.0}               (DEFAULT = 1.0)

`τ=1e-4`, `β=0`, `M=5`, `T=2`, `N_REPS=2`.

In [ ]:
ALPHA_VALUES      = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
ALPHA_ARCH_VALUES = [0.1, 0.5, 1.0, 2.0]

M_GS    = 5
L_GS    = 100
T_GS    = 2.0
TAU_GS  = 1e-4
N_REPS  = 2

teacher_gs = make_wi_particles_resnet(1)
results_gs = {(a, aa): [] for a in ALPHA_VALUES for aa in ALPHA_ARCH_VALUES}

total = len(ALPHA_VALUES) * len(ALPHA_ARCH_VALUES)
done  = 0

for i_a, alpha in enumerate(ALPHA_VALUES):
    for i_aa, alpha_arch in enumerate(ALPHA_ARCH_VALUES):
        cfg = {**DEFAULT_CONFIG_RESNET,
               "alpha": alpha, "alpha_arch": alpha_arch,
               "tau": TAU_GS, "T": T_GS, "gr": 3, "beta": 0.0}
        rep_losses = []
        for rep in range(N_REPS):
            key = jax.random.PRNGKey(rep * 1000 + i_a * 10 + i_aa)
            h   = train_resnet(teacher_gs, M_GS, L_GS, key, cfg, "si")
            losses = h["losses"]
            # Divergence: final loss > 10x initial loss
            final = float(losses[-1]) if np.isfinite(losses[-1]) else np.nan
            if np.isfinite(final) and final > 10 * losses[0]:
                final = np.nan   # marked as diverged
            rep_losses.append(final)
        results_gs[(alpha, alpha_arch)] = rep_losses
        done += 1
        mean_str = f"{np.nanmean(rep_losses):.3e}" if np.any(np.isfinite(rep_losses)) else "DIV"
        print(f"[{done:2d}/{total}]  α={alpha:5.2f}  α_arch={alpha_arch}  loss={mean_str}")

In [ ]:
# Heatmap of log10(loss_mean) — red = diverged
loss_matrix = np.full((len(ALPHA_VALUES), len(ALPHA_ARCH_VALUES)), np.nan)
for i_a, alpha in enumerate(ALPHA_VALUES):
    for i_aa, alpha_arch in enumerate(ALPHA_ARCH_VALUES):
        vals = results_gs[(alpha, alpha_arch)]
        if np.any(np.isfinite(vals)):
            loss_matrix[i_a, i_aa] = np.nanmean(vals)

log_loss = np.where(np.isfinite(loss_matrix), np.log10(loss_matrix + 1e-12), np.nan)

cmap = plt.cm.viridis.copy()
cmap.set_bad(color="crimson")
vmin, vmax = np.nanmin(log_loss), np.nanmax(log_loss)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(log_loss, aspect="auto", origin="lower",
               cmap=cmap, vmin=vmin, vmax=vmax)

ax.set_xticks(range(len(ALPHA_ARCH_VALUES)))
ax.set_xticklabels([str(aa) for aa in ALPHA_ARCH_VALUES])
ax.set_yticks(range(len(ALPHA_VALUES)))
ax.set_yticklabels([str(a) for a in ALPHA_VALUES])
ax.set_xlabel("alpha_arch")
ax.set_ylabel("alpha (learning rate)")
ax.set_title(f"log₁₀(final loss) — ResNet matrix, L={L_GS}, M={M_GS}\n"
             f"(red = diverged, final loss > 10 × initial loss)")

for i in range(len(ALPHA_VALUES)):
    for j in range(len(ALPHA_ARCH_VALUES)):
        val = loss_matrix[i, j]
        if np.isfinite(val):
            brightness = (log_loss[i, j] - vmin) / max(vmax - vmin, 1e-9)
            txt_color  = "white" if brightness < 0.55 else "black"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    fontsize=9, color=txt_color)
        else:
            ax.text(j, i, "DIV", ha="center", va="center",
                    fontsize=9, color="white", fontweight="bold")

plt.colorbar(im, ax=ax, label="log₁₀(loss)")
fig.tight_layout()
plt.show()

# Best config
best_i, best_j = np.unravel_index(np.nanargmin(loss_matrix), loss_matrix.shape)
BEST_ALPHA      = ALPHA_VALUES[best_i]
BEST_ALPHA_ARCH = ALPHA_ARCH_VALUES[best_j]
print(f"\nBest config : alpha={BEST_ALPHA}, alpha_arch={BEST_ALPHA_ARCH}")
print(f"Mean loss       : {loss_matrix[best_i, best_j]:.6f}")

# Compare to default (alpha=5, alpha_arch=1) if in grid
try:
    di = ALPHA_VALUES.index(5.0)
    dj = ALPHA_ARCH_VALUES.index(1.0)
    default_val = loss_matrix[di, dj]
    print(f"DEFAULT (α=5, α_arch=1)     : {'DIV' if np.isnan(default_val) else f'{default_val:.6f}'}")
except ValueError:
    pass

---
## 3. τ sweep (regularization)

With the best `(α, α_arch)`, we test the effect of τ.
Stronger regularization can stabilise training by limiting the growth of z.

In [ ]:
TAU_VALUES = [0.0, 1e-5, 1e-4, 1e-3, 1e-2]
N_REPS_TAU = 3

results_tau = {t: [] for t in TAU_VALUES}

for tau in TAU_VALUES:
    cfg = {**DEFAULT_CONFIG_RESNET,
           "alpha": BEST_ALPHA, "alpha_arch": BEST_ALPHA_ARCH,
           "tau": tau, "T": T_GS, "gr": 3, "beta": 0.0}
    for rep in range(N_REPS_TAU):
        key = jax.random.PRNGKey(rep * 100)
        h   = train_resnet(teacher_gs, M_GS, L_GS, key, cfg, "si")
        final = float(h["losses"][-1])
        results_tau[tau].append(final if np.isfinite(final) else np.nan)
    mean_str = f"{np.nanmean(results_tau[tau]):.4e}" if np.any(np.isfinite(results_tau[tau])) else "DIV"
    print(f"tau={tau:.0e}  loss={mean_str}")

# Best tau
BEST_TAU = TAU_VALUES[int(np.nanargmin([np.nanmean(results_tau[t]) for t in TAU_VALUES]))]
print(f"\nMeilleur τ : {BEST_TAU}")

---
## 4. Full validation — best config vs DEFAULT

We compare the best config found against DEFAULT_CONFIG_RESNET for the
three schemes (vanilla / DA / FA), with `L=100`, `M=5` and `gr=20` (smooth curves).

In [ ]:
BEST_CONFIG = {**DEFAULT_CONFIG_RESNET,
               "alpha":      BEST_ALPHA,
               "alpha_arch": BEST_ALPHA_ARCH,
               "tau":        BEST_TAU,
               "T":          T_GS,
               "gr":         20,
               "beta":       0.0}
DEFAULT_REF = {**DEFAULT_CONFIG_RESNET, "T": T_GS, "gr": 20, "beta": 0.0}

print(f"BEST : alpha={BEST_ALPHA}, alpha_arch={BEST_ALPHA_ARCH}, tau={BEST_TAU}")
print(f"REF  : alpha=5, alpha_arch=1, tau=1e-4")

In [ ]:
N_REPS_VAL = 3
teacher_val = make_wi_particles_resnet(1)

hist_best    = {"vanilla": [], "DA": [], "FA": []}
hist_default = {"vanilla": [], "DA": [], "FA": []}

for rep in range(N_REPS_VAL):
    key = jax.random.PRNGKey(rep * 100)
    for fold, cfg, store in [(0, BEST_CONFIG, hist_best),
                              (10, DEFAULT_REF,  hist_default)]:
        h = train_resnet(teacher_val, M_GS, L_GS,
                         jax.random.fold_in(key, fold),   cfg, "si")
        store["vanilla"].append(h)
        h = train_resnet(teacher_val, M_GS, L_GS,
                         jax.random.fold_in(key, fold+1), cfg, "si",
                         used_loss_fn=loss_fn_da_resnet)
        store["DA"].append(h)
        h = train_resnet(teacher_val, M_GS, L_GS,
                         jax.random.fold_in(key, fold+2), cfg, "si",
                         used_loss_fn=loss_fn_fa_resnet)
        store["FA"].append(h)
    print(f"rep {rep} done")

In [ ]:
def plot_validation_curves(save_pdf=False):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    pairs = [
        (hist_best,    f"Best config (α={BEST_ALPHA}, α_arch={BEST_ALPHA_ARCH}, τ={BEST_TAU})", axes[0]),
        (hist_default, "Default (α=5, α_arch=1, τ=1e-4)",                                           axes[1]),
    ]
    for histories, title, ax in pairs:
        for name, hists in histories.items():
            steps    = np.array(hists[0]["steps"][1:])
            lm       = np.stack([np.array(h["losses"]) for h in hists])
            mean, std = lm.mean(0), lm.std(0)
            color = DEFAULT_COLORS.get(name)
            n_pts = len(steps)
            ax.plot(steps, mean, marker=DEFAULT_MARKERS.get(name, "o"),
                    color=color, label=name, markersize=4,
                    markevery=max(1, n_pts // 5))
            ax.fill_between(steps, np.maximum(mean - std, 1e-12), mean + std,
                            alpha=0.2, color=color)
        ax.set_yscale("log")
        ax.set_xlabel("SGD step")
        ax.set_ylabel("Loss")
        ax.set_title(title, fontsize=10)
        ax.legend()
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"ResNet matrix, L={L_GS}, M={M_GS}", fontsize=12)
    fig.tight_layout()
    if save_pdf:
        fig.savefig("matrix_rn_validation_curves.pdf", format="pdf", bbox_inches="tight")
    plt.show()

plot_validation_curves(save_pdf=False)

---
## 5. Loss vs L — best config vs DEFAULT

We check that the loss decreases with the depth L,
and compare with the default config to quantify the improvement.

In [ ]:
L_VALUES   = [1, 10, 100, 500]
N_REPS_L   = 3

cfg_best_L    = {**BEST_CONFIG,  "gr": 3}
cfg_default_L = {**DEFAULT_REF,  "gr": 3}

res_best    = {"vanilla": [], "DA": [], "FA": []}
res_default = {"vanilla": [], "DA": [], "FA": []}

teacher_L = make_wi_particles_resnet(1)

for L in L_VALUES:
    for store in [res_best, res_default]:
        for name in store:
            store[name].append([])

    for rep in range(N_REPS_L):
        key = jax.random.PRNGKey(rep * 1000 + L)
        for fold, cfg, store in [(0, cfg_best_L, res_best),
                                  (10, cfg_default_L, res_default)]:
            h = train_resnet(teacher_L, M_GS, L,
                             jax.random.fold_in(key, fold), cfg, "si")
            store["vanilla"][-1].append(h["losses"][-1])
            h = train_resnet(teacher_L, M_GS, L,
                             jax.random.fold_in(key, fold+1), cfg, "si",
                             used_loss_fn=loss_fn_da_resnet)
            store["DA"][-1].append(h["losses"][-1])
            h = train_resnet(teacher_L, M_GS, L,
                             jax.random.fold_in(key, fold+2), cfg, "si",
                             used_loss_fn=loss_fn_fa_resnet)
            store["FA"][-1].append(h["losses"][-1])
    print(f"L={L} done")

In [ ]:
def plot_loss_vs_L_comparison(save_pdf=False):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    datasets = [
        (res_best,    f"Best config (α={BEST_ALPHA}, α_arch={BEST_ALPHA_ARCH}, τ={BEST_TAU})", axes[0]),
        (res_default, "Default (α=5, α_arch=1, τ=1e-4)",                                          axes[1]),
    ]
    for results, title, ax in datasets:
        for name, reps_per_L in results.items():
            means = [np.nanmean(r) for r in reps_per_L]
            stds  = [np.nanstd(r)  for r in reps_per_L]
            ax.errorbar(L_VALUES, means, yerr=stds,
                        marker=DEFAULT_MARKERS.get(name, "o"),
                        color=DEFAULT_COLORS.get(name),
                        label=name, capsize=3)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Depth L")
        ax.set_ylabel("Final loss")
        ax.set_title(title, fontsize=10)
        ax.legend()
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"Final loss vs L — ResNet matrix, M={M_GS}, teacher WI, init SI", fontsize=12)
    fig.tight_layout()
    if save_pdf:
        fig.savefig("matrix_rn_loss_vs_L.pdf", format="pdf", bbox_inches="tight")
    plt.show()

plot_loss_vs_L_comparison(save_pdf=False)